# Phase 1 — Data exploration and validated case foundation

This executable submission notebook demonstrates the supplied customer, account, and transaction data; validates relationship integrity; identifies auditable screening patterns; and exercises the canonical Pydantic/DataLoader/audit implementation in `starter/src/foundation_sar.py`.


In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "data").is_dir() and (cwd / "src").is_dir():
    STARTER = cwd
elif cwd.name == "notebooks":
    STARTER = cwd.parent
else:
    STARTER = cwd / "starter"
sys.path.insert(0, str(STARTER))

from src.foundation_sar import (
    AccountData,
    DataLoader,
    load_csv_data,
)

DATA = STARTER / "data"
OUTPUTS = STARTER / "outputs" / "audit_logs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f"Using data directory: {DATA}")

Using data directory: D:\Udacity Course Agentic AI For Financial Services\Course 1\project\starter\data


## Source data quality and relationships

`ssn_last_4` is loaded as text to retain leading zeroes. Blank optional values remain acceptable at the schema seam, while identifiers, dates, amounts, balances, and relationships are validated.

In [2]:
customers_df, accounts_df, transactions_df = load_csv_data(DATA)

summary = pd.DataFrame(
    {
        "dataset": ["customers", "accounts", "transactions"],
        "rows": [len(customers_df), len(accounts_df), len(transactions_df)],
        "duplicate_ids": [
            customers_df["customer_id"].duplicated().sum(),
            accounts_df["account_id"].duplicated().sum(),
            transactions_df["transaction_id"].duplicated().sum(),
        ],
    }
)

orphan_accounts = (~accounts_df["customer_id"].isin(customers_df["customer_id"])).sum()
orphan_transactions = (~transactions_df["account_id"].isin(accounts_df["account_id"])).sum()
display(summary)
print({"orphan_accounts": int(orphan_accounts), "orphan_transactions": int(orphan_transactions)})

,dataset,rows,duplicate_ids
0,customers,150,0
1,accounts,178,0
2,transactions,4268,0


{'orphan_accounts': 0, 'orphan_transactions': 0}


## Deterministic screening evidence

The following checks identify potentially relevant patterns without reading scenario hints embedded in transaction IDs. A repeated series of cash deposits immediately below the $10,000 CTR threshold can warrant structuring review, but intent must be assessed from the complete facts.

In [3]:
cash_band = transactions_df[
    transactions_df["transaction_type"].eq("Cash_Deposit")
    & transactions_df["amount"].between(9000, 10000, inclusive="left")
].copy()

activity_by_customer = (
    transactions_df.merge(accounts_df[["account_id", "customer_id"]], on="account_id")
    .groupby("customer_id")
    .agg(transaction_count=("transaction_id", "count"), total_amount=("amount", "sum"))
    .sort_values(["transaction_count", "total_amount"], ascending=False)
)

print({
    "cash_deposits_9000_to_under_10000": len(cash_band),
    "customers_with_50_or_more_transactions": int((activity_by_customer["transaction_count"] >= 50).sum()),
})
display(activity_by_customer.head(10))

{'cash_deposits_9000_to_under_10000': 18, 'customers_with_50_or_more_transactions': 20}


,transaction_count,total_amount
customer_id,,
CUST_0111,121,386834.91
CUST_0057,101,286342.62
CUST_0053,94,274025.18
CUST_0062,90,380633.78
CUST_0118,79,222070.26
CUST_0129,79,216693.67
CUST_0135,76,182852.90
CUST_0035,69,129261.54
CUST_0143,68,171150.38


## Canonical loading, case assembly, and validation

`DataLoader.load` validates every CSV row once, checks foreign-key relationships, builds reusable indexes, and returns stable case snapshots for the application.

In [4]:
audit_path = OUTPUTS / "phase1_notebook.jsonl"
if audit_path.exists():
    audit_path.unlink()

loader = DataLoader.load(DATA, audit_path)
case = loader.get_case("CUST_0053")

print(loader.summary().model_dump())
print({
    "case_id": case.case_id,
    "customer_id": case.customer.customer_id,
    "accounts": len(case.accounts),
    "transactions": len(case.transactions),
    "created_at": case.created_at.isoformat(),
})

try:
    AccountData(
        account_id="INVALID",
        customer_id="CUST_0053",
        account_type="Checking",
        opening_date="2025-01-01",
        current_balance=-1,
        average_monthly_balance=0,
        status="Active",
    )
except ValueError as exc:
    print(f"Expected validation failure: {exc.errors()[0]['msg']}")

{'customer_count': 150, 'account_count': 178, 'transaction_count': 4268}
{'case_id': 'CASE-c5905753-605e-410d-8c7f-0f3f4ce0a392', 'customer_id': 'CUST_0053', 'accounts': 3, 'transactions': 94, 'created_at': '2026-08-24T14:03:46.201267+00:00'}
Expected validation failure: Input should be greater than or equal to 0


## Phase 1 result

- All 150 customers, 178 accounts, and 4,268 transactions load through validated schemas.
- No duplicate primary IDs or orphan account/transaction relationships are present.
- The dataset contains a concentrated under-$10,000 cash-deposit pattern suitable for risk-agent review.
- Case identity is distinct from customer identity and timestamped in UTC.
- Data loading and case creation are recorded in append-only JSONL audit output.

The next phase owns financial-crime interpretation and classification.